In [1]:
import pandas as pd
import geopandas as gpd
import datetime

import os
import sys
sys.path.append('..')

from sqlalchemy import create_engine, text

from config import RUTA_UNIDAD_ONE_DRIVE
from config import RUTA_LOCAL_ONE_DRIVE
from config import POSTGRES_UTEA

USER_DB = POSTGRES_UTEA['USER']
PASS_DB = POSTGRES_UTEA['PASSWORD']
HOST_DB = POSTGRES_UTEA['HOST']
PORT_DB = POSTGRES_UTEA['PORT']
NAME_DB = POSTGRES_UTEA['DATABASE']

In [14]:
# funcion que retorna el encale para conectar a base de datos
def get_engine():
    return create_engine(
        f"postgresql+psycopg2://{POSTGRES_UTEA['USER']}:{POSTGRES_UTEA['PASSWORD']}@{POSTGRES_UTEA['HOST']}:{POSTGRES_UTEA['PORT']}/{POSTGRES_UTEA['DATABASE']}"
    )

# obtener todo el catastro de la base datos
def get_catastro():
    engine = get_engine()
    try:
        query = """
            SELECT * FROM catastro_iag.catastro
        """
        gdf = gpd.read_postgis(query, engine, geom_col='geom')
        return gdf
    except Exception as e:
        print(f"❌ Error en get_catastro(): {e}")
        return gpd.GeoDataFrame()
    return None

# carga el nombre del cañero, cod_tra y inst segun xlsx de GRUPO_TRABAJO
# usa como relacion el codigo cañero (unidad_03)
def actualizar_catastro_desde_df(df):
    """
    df: DataFrame que contiene unidad_03, cod_tra, delegado e inst
    engine: El objeto engine de tu función get_engine()
    """
    engine = get_engine()
    # Nombre de la tabla de destino y tabla temporal
    tabla_destino = "catastro"
    schema_destino = "catastro_iag"
    tabla_temp = "temp_update_catastro"

    with engine.begin() as conn:
        # 0. Subir el DataFrame a una tabla temporal
        # Solo subimos las columnas necesarias para la actualización
        cols_interes = ['unidad_03', 'unidad_04', 'cod_tra', 'inst']
        df[cols_interes].to_sql(tabla_temp, conn, if_exists='replace', index=False)
        
        # 1. Limpieza inicial: Poner en NULL los campos en la tabla de destino
        # Esto afecta a TODA la tabla. Si solo quieres limpiar los que vas a actualizar, 
        # podrías omitir este paso, ya que el UPDATE del paso 2 sobrescribe los valores.
        print("Limpiando campos antiguos (seteando a NULL)...")
        query_limpieza = text(f"""
            UPDATE {schema_destino}.{tabla_destino} 
            SET unidad_04 = NULL, 
                cod_tra = NULL, 
                inst = NULL;
        """)
        conn.execute(query_limpieza)

        # 2. Ejecutar el UPDATE masivo con un JOIN
        # Usamos la sintaxis de PostgreSQL: UPDATE ... FROM
        query = text(f"""
            UPDATE {schema_destino}.{tabla_destino} AS destino
            SET 
                unidad_04 = temporal.unidad_04,
                cod_tra = temporal.cod_tra,
                inst = temporal.inst
            FROM {tabla_temp} AS temporal
            WHERE destino.unidad_03 = temporal.unidad_03;
        """)
        
        result = conn.execute(query)
        print(f"Actualización completada. Filas afectadas: {result.rowcount}")

        # 3. Limpiar: Borrar la tabla temporal
        conn.execute(text(f"DROP TABLE {tabla_temp};"))

# genera resumen para calidar campos y condiciones de los datos
def generar_resumen_geodataframe(gdf):
    total_registros = len(gdf)
    ahora = datetime.datetime.now()
    anio_actual = ahora.year
    
    # 1. Conteo de nulos y vacíos (incluye strings de puros espacios)
    resumen_nulos = gdf.replace(r'^\s*$', pd.NA, regex=True).isnull().sum()
    lineas_nulos = "\n".join([f"- {col}: {cant}" for col, cant in resumen_nulos.items() if cant > 0])
    if not lineas_nulos:
        lineas_nulos = "No se encontraron valores nulos o vacíos."

    # 2. Función auxiliar para frecuencias ORDENADAS
    def get_freq_text(col_name, ordenar_por_valor=True):
        if col_name not in gdf.columns:
            return f"  [!] Columna '{col_name}' no encontrada.\n"
        
        counts = gdf[col_name].value_counts()
        if ordenar_por_valor:
            counts = counts.sort_index()
            
        percents = (counts / total_registros) * 100
        
        texto = ""
        for index, val in counts.items():
            texto += f"  * {index}: {val} registros ({percents[index]:.2f}%)\n"
        return texto

    # --- NUEVA LÓGICA: Validación de Fechas (Año Actual) ---
    conteo_fs_actual = 0
    if 'fs' in gdf.columns:
        fechas_siembra = pd.to_datetime(gdf['fs'], errors='coerce')
        conteo_fs_actual = (fechas_siembra.dt.year == anio_actual).sum()
    # -------------------------------------------------------

    # 3. Validación de SOCA
    validacion_soca = ""
    if 'soca' in gdf.columns and 'fs' in gdf.columns:
        fechas_siembra = pd.to_datetime(gdf['fs'], errors='coerce')
        soca_calculada = (anio_actual - fechas_siembra.dt.year) - 1
        
        mascara_validos = fechas_siembra.notnull() & gdf['soca'].notnull()
        coinciden = (gdf.loc[mascara_validos, 'soca'] == soca_calculada[mascara_validos]).sum()
        discrepantes = mascara_validos.sum() - coinciden
        
        validacion_soca = (f"Validación de Soca (vs año {anio_actual}):\n"
                           f"  - Registros que cumplen la fórmula: {coinciden}\n"
                           f"  - Registros con error de cálculo: {discrepantes}\n"
                           f"  - (Fórmula usada: {anio_actual} - año_siembra - 1)")
    else:
        validacion_soca = "No se pudo validar 'soca' (faltan columnas 'soca' o 'fs')."

    # 4. Construcción del Texto Final
    resumen_texto = f"""
==================================================
        RESUMEN DE GESTIÓN DE LOTES
==================================================
Total de registros en el set: {total_registros}

REGISTROS NULOS O VACÍOS:
{lineas_nulos}

DISTRIBUCIÓN DE CATEGORÍAS (Ordenadas):

Variedades:
{get_freq_text('variedad', ordenar_por_valor=False)} 
Soca (Nivel actual):
{get_freq_text('soca', ordenar_por_valor=True)}
Tipo de Cultivo:
{get_freq_text('cultivo', ordenar_por_valor=False)}
Financiamiento:
{get_freq_text('financia', ordenar_por_valor=False)}

VALIDACIÓN DE FECHAS:
- Registros con fecha de siembra (fs) del año actual ({anio_actual}): {conteo_fs_actual}

VALIDACIÓN LÓGICA:
{validacion_soca}
==================================================
    """
    return resumen_texto

# calcula el id (0000 + unidad_01 + 0000 + unidad_05) y calcula Soca en base a fs (fs no nulos)
def ejecutar_calculos_en_db():
    """
    Ejecuta el cálculo de ID y SOCA directamente en PostgreSQL.
    """
    engine = get_engine()
    anio_actual = datetime.datetime.now().year
    
    # Definimos las consultas SQL
    # 1. Concatenación de ID: Convertimos a TEXT y aseguramos formato
    # 2. Cálculo de SOCA: Usamos EXTRACT(YEAR FROM campo)
    
    query_update = text(f"""
        UPDATE catastro_iag.catastro
        SET 
            -- Generar ID: '0000' + unidad_01 + '0000' + unidad_05
            id = '0000' || CAST(unidad_01 AS TEXT) || '0000' || CAST(unidad_05 AS TEXT),
            
            -- Calcular SOCA: (Año Actual - Año de fs) - 1
            soca = ({anio_actual} - EXTRACT(YEAR FROM fs)) - 1
        WHERE fs IS NOT NULL; -- Solo actualiza donde hay fecha para evitar errores
    """)

    try:
        with engine.begin() as conn:
            result = conn.execute(query_update)
            print(f"Cálculos finalizados en la base de datos.")
            print(f"Filas actualizadas: {result.rowcount}")
    except Exception as e:
        print(f"Error al ejecutar la actualización en DB: {e}")

In [3]:
# ruta de _DATOS_PYTHON
PATH_COMPLETO = os.path.join(RUTA_UNIDAD_ONE_DRIVE, RUTA_LOCAL_ONE_DRIVE)

# ruta de GRUPOS_TRABAJO
PATH_GRUPOS_TRABAJO = RUTA_UNIDAD_ONE_DRIVE + r'\Ingenio Azucarero Guabira S.A\UTEA - SEMANAL - AVANCE COSECHA\2026\DATA\GRUPO_TRABAJO.xlsx'
# cargar en dataframe los grupos de trabajo
DF_GRUPOS_TRABAJO = pd.read_excel(PATH_GRUPOS_TRABAJO, sheet_name='2026')

In [4]:
# dict para cambio de nombre de columnas de GRUPO_TRABAJO
nuevos_nombres = {
    'CODIGO CAÑERO': 'unidad_03',
    'NOMBRE CAÑERO': 'unidad_04',
    'GRUPO TRABAJO': 'cod_tra',
    'INS': 'inst',
    'DELEGADO': 'delegado'
}
# cambio de nombre de columnas
DF_GRUPOS_TRABAJO = DF_GRUPOS_TRABAJO.rename(columns=nuevos_nombres)
# actualziar campos en catastro
actualizar_catastro_desde_df(DF_GRUPOS_TRABAJO)

Limpiando campos antiguos (seteando a NULL)...
Actualización completada. Filas afectadas: 13384


In [12]:
# optener todos los datos de catastro
gdf_catrastro = get_catastro()

# filtar caña habilitada para zafra actual
gdf_catastro_canha = gdf_catrastro[(gdf_catrastro['unidad_03'] > 0) & 
                                   ((gdf_catrastro['cultivo'] == 'canha') | (gdf_catrastro['cultivo'] == 'semilla')) & 
                                   (gdf_catrastro['zafra'] == 2026)]
gdf_catastro_canha['area'].sum()

57462.946456958016

In [13]:
# Ejemplo de uso:
texto_resultado = generar_resumen_geodataframe(gdf_catastro_canha)
print(texto_resultado)


        RESUMEN DE GESTIÓN DE LOTES
Total de registros en el set: 10451

REGISTROS NULOS O VACÍOS:
- unidad_04: 115
- fc: 1655
- cod_tra: 115
- fci: 387
- delagado: 10451
- inst: 115

DISTRIBUCIÓN DE CATEGORÍAS (Ordenadas):

Variedades:
  * UCG9020: 7069 registros (67.64%)
  * RBB7726: 1119 registros (10.71%)
  * CITTCA8522: 814 registros (7.79%)
  * UCG9610: 331 registros (3.17%)
  * NA5626: 322 registros (3.08%)
  * RB2: 270 registros (2.58%)
  * CITTCA0563: 221 registros (2.11%)
  * SP835073: 159 registros (1.52%)
  * OTRAS: 37 registros (0.35%)
  * RB4: 36 registros (0.34%)
  * RBN02: 18 registros (0.17%)
  * NN5: 16 registros (0.15%)
  * SP701143: 15 registros (0.14%)
  * CITTCA001: 9 registros (0.09%)
  * NA732596: 7 registros (0.07%)
  * NA1: 5 registros (0.05%)
  * SP931213: 2 registros (0.02%)
  * NA2: 1 registros (0.01%)
 
Soca (Nivel actual):
  * 0.0: 721 registros (6.90%)
  * 1.0: 1264 registros (12.09%)
  * 2.0: 914 registros (8.75%)
  * 3.0: 1266 registros (12.11%)
  * 4

In [9]:
# Ejecución
ejecutar_calculos_en_db()

Cálculos finalizados en la base de datos.
Filas actualizadas: 12791
